# 0. les biblios

In [ ]:
import pandas as pd

import re

import matplotlib.pyplot as plt

import seaborn as sns

import numpy as np

# 1. Charger inventory_month_end.csv

In [ ]:
df = pd.read_csv("inventory_month_end.csv")

In [ ]:
print("Dimensions :", df.shape)

print("\nTypes de colonnes :")

df.info()

In [ ]:
print("\nPremières lignes :")

display(df.head())

- 502 lignes et 11 colonnes.

- Une ligne = l'état des stocks d'une catégorie d'article, sur un site, pour un mois donné (clôture

mensuelle d'inventaire).

- Colonnes de flux : stock d'ouverture, quantité reçue, quantité sortie, usage interne documenté, perte/

déchet documenté, stock de clôture.

- Plusieurs colonnes numériques sont en texte : mêmes signes qu'avant.

# 2. Doublons

### a- Doublons de lignes

In [ ]:
print("Lignes strictement dupliquées :", df.duplicated().sum())

display(df[df.duplicated(keep=False)].sort_values(['facility_code', 'reporting_month']))

### b- Doublons sur la clé métier



La clé logique ici est (site, mois, catégorie d'article) : un site ne peut pas avoir deux clôtures de

stock différentes pour la même catégorie le même mois.

In [ ]:
cle = ['facility_code', 'reporting_month', 'item_category']

print("Couples (site, mois, catégorie) dupliqués :", df.duplicated(subset=cle).sum())

display(df[df.duplicated(subset=cle, keep=False)].sort_values(cle))

- Solution

In [ ]:
df = df.drop_duplicates(keep='first')

df = df.drop_duplicates(subset=cle, keep='first')



print("Nombre de lignes après suppression :", df.shape[0])

print("Doublons restants :", df.duplicated(subset=cle).sum())

### c- Doublons de colonnes

In [ ]:
print("Noms de colonnes en double :", df.columns[df.columns.duplicated()].tolist())

print("Colonnes au contenu identique :", df.columns[df.T.duplicated()].tolist())

# 3. Formats incohérents

In [ ]:
print(df.dtypes)

### - Vérifier le format de facility_code et reporting_month <--->

In [ ]:
print("Format facility_code :", df['facility_code'].str.replace(r'\d', '9', regex=True).unique())



dates_test = pd.to_datetime(df['reporting_month'], errors='coerce')

print("reporting_month non convertibles :", dates_test.isna().sum())

→ Formats homogènes, aucune date ne bloque la conversion. On convertit directement :

In [ ]:
df['reporting_month'] = pd.to_datetime(df['reporting_month'])

print(df['reporting_month'].dtype)

print("Période couverte :", df['reporting_month'].min(), "->", df['reporting_month'].max())

### - Vérifier le format des colonnes de flux (numériques) <--->

In [ ]:
cols_flux = ['opening_stock', 'received_quantity', 'outgoing_quantity',

             'documented_internal_use', 'documented_waste', 'closing_stock']



for col in cols_flux:

    s = df[col]

    non_num = s[pd.to_numeric(s, errors='coerce').isna() & s.notna()]

    print(col, "| non convertibles :", len(non_num), "| exemples :", non_num.unique()[:6])

**Constat, deux problèmes différents :**

- `opening_stock` et `closing_stock` : l'unité est parfois recopiée dans la valeur (`"14.55 sealed_lot"`),

alors qu'elle est déjà présente dans la colonne `unit` — même redondance que dans `goods_receipts.csv`.

- `received_quantity`, `outgoing_quantity`, `documented_waste` : quelques virgules décimales.

- Solution

In [ ]:
for col in cols_flux:

    df[col] = (df[col].astype(str)

               .str.replace(',', '.', regex=False)

               .str.replace(r'\s+[a-z_]+$', '', regex=True)  # enlève l'unité recopiée

               .str.strip())

    df[col] = pd.to_numeric(df[col], errors='coerce')



print(df[cols_flux].dtypes)

print("\nValeurs non convertibles restantes :")

print(df[cols_flux].isna().sum())

### - Vérifier le format de item_category et unit (catégorielles)

In [ ]:
print(df['item_category'].value_counts())

print()

print(df['unit'].value_counts())

→ Déjà propres, pas de souci de casse.

### - Vérifier le format de inventory_status (catégorielle) <--->

In [ ]:
print(df['inventory_status'].value_counts(dropna=False))

**Constat :** `closed`/`CLOSED` et `provisional`/`PROVISIONAL` — casse incohérente classique.

- Solution

In [ ]:
df['inventory_status'] = df['inventory_status'].str.strip().str.lower()

print(df['inventory_status'].value_counts())

# 4. Valeurs manquantes

In [ ]:
print("Valeurs manquantes par colonne :")

print(df.isna().sum())

→ **Aucune valeur manquante** dans ce fichier, une première sur le projet. Rien à traiter ici.

# 5. Cohérence des flux de stock (vérification métier)

Avant de continuer, on vérifie si la logique comptable du fichier est cohérente, ce qui peut aider

à repérer d'éventuelles erreurs de saisie cachées (pas visibles via les vérifications de format classiques).

### a. La clôture d'un mois correspond-elle à l'ouverture du mois suivant ?



Hypothèse : pour un même site et une même catégorie, le stock de clôture d'un mois devrait être repris

comme stock d'ouverture du relevé suivant.

In [ ]:
df = df.sort_values(['facility_code', 'item_category', 'reporting_month']).reset_index(drop=True)

df['next_opening'] = df.groupby(['facility_code', 'item_category'])['opening_stock'].shift(-1)



comparaison = df.dropna(subset=['next_opening'])

ecart = (comparaison['closing_stock'] - comparaison['next_opening']).round(2)

print("Correspondance exacte :", (ecart.abs() < 0.01).mean().round(4) * 100, "% des cas")

print(ecart.describe())

→ **Confirmé à 99.6%** : la clôture d'un mois redevient bien l'ouverture du relevé suivant. C'est

cohérent, ça confirme que les colonnes `opening_stock`/`closing_stock` sont fiables et bien chaînées dans

le temps (utile à savoir avant de faire une analyse temporelle, mais rien à corriger ici).

In [ ]:
df = df.drop(columns=['next_opening'])

### b. Le calcul du flux (ouverture + reçu - sorti - usage interne - déchet) donne-t-il la clôture ?

In [ ]:
df['calc_closing'] = (df['opening_stock'] + df['received_quantity'] - df['outgoing_quantity']

                       - df['documented_internal_use'] - df['documented_waste'])

ecart_flux = (df['calc_closing'] - df['closing_stock']).round(2)

print(ecart_flux.describe())

print("Écart nul dans", (ecart_flux.abs() < 0.01).mean().round(4) * 100, "% des cas seulement")

**Constat :** contrairement à la vérification précédente, ce calcul ne correspond presque jamais à

la clôture réelle. **On ne traite pas ça comme une erreur à corriger** : rien n'indique que la formule

attendue soit exactement celle-là — il existe probablement d'autres mouvements de stock non

enregistrés dans ce fichier (transferts entre sites, ajustements du fichier `inventory_adjustments.csv`

qu'on a déjà nettoyé, par exemple). On garde toutes les colonnes telles quelles et on documente cette

observation, plutôt que de forcer artificiellement une cohérence qu'on ne peut pas garantir.

In [ ]:
df = df.drop(columns=['calc_closing'])

# 6. Gestion des variables catégorielles

### a. Création de variables temporelles

In [ ]:
df['year'] = df['reporting_month'].dt.year

df['month'] = df['reporting_month'].dt.month

display(df.head())

### b. Variables ordonnées

`inventory_status` (`closed` / `provisional` / `revised`) pourrait sembler ordonné (un relevu passe de

`provisional` à `closed`, parfois `revised` après coup), mais ce n'est pas un ordre de grandeur, plutôt un

statut de cycle de vie — on la garde nominale par prudence, comme les autres catégories du fichier.

### c. Encodage des variables nominales : reporté après le merge

Comme pour les autres fichiers logistiques, on garde les catégories nettoyées en texte

(`item_category`, `unit`, `inventory_status`). L'encodage One-Hot sera fait une seule fois, après le merge

avec `goods_receipts.csv`, `inventory_adjustments.csv`, `erp_facilities.csv`.

In [ ]:
df_stats = df.copy()

print(df_stats.dtypes)

display(df_stats.head())

# 7. Voir les outliers (colonnes numériques)

In [ ]:
cols_num = ['opening_stock', 'received_quantity', 'outgoing_quantity',

            'documented_internal_use', 'documented_waste', 'closing_stock']

display(df_stats[cols_num].describe().T)

In [ ]:
plt.figure(figsize=(16, 8))

for i, col in enumerate(cols_num, 1):

    plt.subplot(2, 3, i)

    sns.boxplot(y=df_stats[col], color="skyblue")

    plt.title(col)

plt.tight_layout()

plt.show()

In [ ]:
for col in cols_num:

    q1, q3 = df_stats[col].quantile([0.25, 0.75])

    iqr = q3 - q1

    n = ((df_stats[col] < q1 - 1.5*iqr) | (df_stats[col] > q3 + 1.5*iqr)).sum()

    print(f"{col:28s} min={df_stats[col].min():7.1f} max={df_stats[col].max():7.1f} -> {n} outliers")

**Constat :** comme pour `received_quantity` dans `goods_receipts.csv`, ces colonnes mélangent

plusieurs unités (`sealed_lot`, `case`, `kit`, `pallet`) aux échelles très différentes.

In [ ]:
display(df_stats.groupby('unit')[cols_num].mean().round(1))

→ Les valeurs extrêmes s'expliquent par l'unité (les `pallet` ont naturellement des quantités plus

hautes). Aucune valeur négative ni impossible détectée. **On conserve toutes les valeurs.**

# 8. Histogrammes et distributions

In [ ]:
plt.figure(figsize=(16, 8))

for i, col in enumerate(cols_num, 1):

    plt.subplot(2, 3, i)

    sns.histplot(df_stats[col], kde=True, bins=25, color='skyblue')

    plt.title(f"Distribution : {col}")

plt.tight_layout()

plt.show()



print(df_stats[cols_num].skew().round(2))

→ Toutes les colonnes de flux sont fortement asymétriques vers la droite (skew élevé), pour la même

raison que `received_quantity` : le mélange d'unités crée une longue traîne. La médiane est plus

représentative que la moyenne pour résumer ces colonnes.

In [ ]:
plt.figure(figsize=(14, 4))



plt.subplot(1, 3, 1)

sns.countplot(y='item_category', data=df_stats,

              order=df_stats['item_category'].value_counts().index, color='skyblue')

plt.title("Répartition par catégorie d'article")



plt.subplot(1, 3, 2)

sns.countplot(y='unit', data=df_stats,

              order=df_stats['unit'].value_counts().index, color='skyblue')

plt.title("Répartition des unités")



plt.subplot(1, 3, 3)

sns.countplot(y='inventory_status', data=df_stats,

              order=df_stats['inventory_status'].value_counts().index, color='skyblue')

plt.title("Répartition des statuts")



plt.tight_layout()

plt.show()

# 9. Corrélations

In [ ]:
matrice_spearman = df_stats[cols_num].corr(method='spearman')

matrice_pearson = df_stats[cols_num].corr(method='pearson')



display(matrice_spearman.round(2))

In [ ]:
plt.figure(figsize=(8, 6))

sns.heatmap(matrice_spearman, annot=True, cmap='coolwarm', vmin=-1, vmax=1, fmt=".2f", linewidths=0.5)

plt.title("Matrice de corrélation de Spearman")

plt.tight_layout()

plt.show()

In [ ]:
sns.pairplot(df_stats[cols_num], kind='scatter', plot_kws={'alpha': 0.3, 'color': 'teal', 's': 15})

plt.suptitle("Nuages de points croisés", y=1.02)

plt.show()

### - Analyse des corrélations



- **`opening_stock` ↔ `closing_stock` (fortement positive)** : logique, un site avec un gros stock au

début du mois a généralement un gros stock à la fin aussi — confirme la cohérence trouvée à l'étape 5a.

- **`received_quantity`, `outgoing_quantity`, `documented_waste`, `documented_internal_use`** : corrélations

plus faibles entre elles, chaque flux semble varier indépendamment des autres.

- Comme toujours, **corrélation n'est pas causalité** : un stock d'ouverture élevé ne "cause" pas un stock

de clôture élevé, les deux reflètent simplement le volume d'activité global de ce site pour cette

catégorie d'article.

# 10. Export du jeu de données nettoyé

In [ ]:
df_stats.to_csv("inventory_month_end_clean.csv", index=False)



print("Dimensions finales :", df_stats.shape)

print("Valeurs manquantes restantes :", df_stats.isna().sum().sum())

# 11. Synthèse du nettoyage



| Problème identifié | Colonnes concernées | Traitement appliqué |

|---|---|---|

| Doublons stricts (2) et doublons de clé (site, mois, catégorie) | toutes | suppression, on garde la 1ère occurrence |

| Unité recopiée dans la valeur | `opening_stock`, `closing_stock` | suppression de l'unité redondante |

| Virgule décimale | `received_quantity`, `outgoing_quantity`, `documented_waste` | `,` → `.` |

| Casse incohérente | `inventory_status` | normalisation en minuscules |

| Valeurs manquantes | — | aucune trouvée |

| Cohérence temporelle (clôture = ouverture du mois suivant) | `opening_stock` / `closing_stock` | vérifiée à 99.6%, confirmée fiable, rien à corriger |

| Cohérence du flux (ouverture + reçu - sorti - usages - déchets ≠ clôture) | colonnes de flux | observation documentée, **non corrigée** (mouvements probablement non tous tracés dans ce fichier) |

| Encodage des catégorielles | toutes | **reporté** : fait une seule fois après le merge des fichiers logistiques |



**Biais introduits et assumés :**

- ne pas "forcer" la formule de flux à être cohérente évite d'inventer une correction sur des données

qu'on ne comprend pas complètement — mieux vaut documenter l'incohérence que la masquer artificiellement ;

- comme pour les fichiers précédents, on garde la 1ère occurrence des doublons après vérification qu'ils

étaient identiques.